In [18]:
survey_control_guid = "a236b79f-6d2e-4f23-ad92-d018cb518346"
guid_info = "7c0eea07-99a1-458e-a315-2b67cc9981ad"
guid_topics_dcm = "2ecf3ce4-aede-4516-a049-5d274659e5ec"
guid_genres_dcm = "4a30eb82-dc77-47c0-99ab-98d4ffb9759f"
guid_tags = "73df38cd-31aa-43e5-b05b-6e973ab35295"

In [19]:
from src.runtime_constants import rr
import io
from src.db import DbConnection
import pandas as pd

@staticmethod
def save_df_to_external_data(conn: DbConnection, df: pd.DataFrame, guid):
    buf = io.BytesIO()
    df.to_feather(buf)
    buf.seek(0)
    c = buf.read()
    conn.query("""
    DELETE FROM external_data.dms WHERE guid = %s
    """, [guid])
    conn.query("""
    INSERT INTO external_data.dms (guid, data) VALUES (%s, %s)
    """, [guid, c])

conn = DbConnection(rr.ENV_PROD, True)

rows = conn.select_all("""
SELECT si.*
  FROM survey_data.survey_meta sm,
       survey_data.survey_info si
 WHERE sm.survey_control_guid = %s
   AND si.survey_meta_id = sm.id
""", [survey_control_guid])
df = pd.DataFrame.from_records(rows)
save_df_to_external_data(conn, df, guid_info)

rows = conn.select_all("""
SELECT dcm.*
  FROM survey_data.survey_meta sm,
       survey_data.survey_genres_dcm dcm
 WHERE sm.survey_control_guid = %s
   AND dcm.survey_meta_id = sm.id
""", [survey_control_guid])
df = pd.DataFrame.from_records(rows)
save_df_to_external_data(conn, df, guid_genres_dcm)

rows = conn.select_all("""
SELECT t.*
  FROM survey_data.survey_meta sm,
       survey_data.survey_tags t
 WHERE sm.survey_control_guid = %s
   AND t.survey_meta_id = sm.id
""", [survey_control_guid])
df = pd.DataFrame.from_records(rows)
save_df_to_external_data(conn, df, guid_tags)

conn.commit()
conn.close()

In [20]:
conn.close()

In [21]:
df

,id,survey_meta_id,tag_id,tag_value,survey_instance
0,1,1,platforms__mobile,1,33cbe449-af6a-484b-a521-6a4d08f73454
1,2,1,platforms__playstation,1,33cbe449-af6a-484b-a521-6a4d08f73454
2,3,1,platforms__pc,1,33cbe449-af6a-484b-a521-6a4d08f73454
3,4,1,genre__rpg,66,33cbe449-af6a-484b-a521-6a4d08f73454
4,5,1,genre__match,100,33cbe449-af6a-484b-a521-6a4d08f73454
...,...,...,...,...,...
121361,121362,1,multiplayer__multi,4,8ec75988-2510-4d41-b5da-456e237c2574
121362,121363,1,multiplayer__single,3,8ec75988-2510-4d41-b5da-456e237c2574
121363,121364,1,activities__working,1,8ec75988-2510-4d41-b5da-456e237c2574
121364,121365,1,activities__parenting,1,8ec75988-2510-4d41-b5da-456e237c2574
